Kafka orders -> partitioned Parquet lake, with a checkpoint. Bad rows go to quarantine.

In [1]:
import sys, time

In [2]:
%run order.ipynb

In [3]:
seconds = 1

In [4]:
%run spark_common.ipynb

D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar,D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar,D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar,D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar;D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar;D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar;D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar
4.3.0


In [5]:
spark = get_spark("stream-job")

In [6]:
orders = read_orders(spark)

In [ ]:
def start(df, path, partition_by=None):
    writer = (df.writeStream.format("parquet")
              .option("path", f"lake/{path}")
              .option("checkPointLocation", f"checkpoints/{path}")
              )
    if partition_by:
        writer = writer.partitionBy(partition_by)
    if seconds:
        return writer.trigger(processingTime="5 seconds").start()
    return writer.trigger(availableNow=True).start()

In [ ]:
good = orders.filter("is_valid").drop("is_valid", "ts")
bad = orders.filter("NOT is_valid").select("partition", "offset", "order_id", "qty", "city")
queries = [start(good, "orders", "order_date"), start(bad, "bad_orders")]

if seconds:
    end = time.time() + seconds
    while time.time() < end:
        time.sleep(10)
        p = queries[0].lastProgress
        if p:
            print(f"batch {p['batchId']}: {p['numInputRows']} new Kafka records", flush=True)
    for q in queries:
        q.stop()
else:
    for q in queries:
        q.awaitTermination()
read = sum([p["numInputRows"] for p in queries[0].recentProgress])
print(f"read {read} new Kafka records in this run")
print("orders in the lake: ", spark.read.parquet("lake/orders").count())
print("rows in quaraantine: ", spark.read.parquet("lake/bad_orders").count())

In [ ]:
# Run check
# > papermill stream_job.ipynb output_stream_job.ipynb -p seconds 5